can i read pdf ??

In [5]:
from pathlib import Path
pdf_path = Path("../requirement/compensation-policy.pdf")
pdf_path.exists()

True

Read the PDF

In [6]:
from pypdf import PdfReader
reader = PdfReader(pdf_path)
len(reader.pages)
# tells us how many pages the PDF contains

page = reader.pages[0]
text = page.extract_text()
print(text)
#pdf to text 

 
Page 1 of 18 
 
 
Approved by CSCM: March 20, 2026 
 
Customer Compensation Policy 
Introduction 
 
The Compensation Policy (the Policy) of the Bank reflects the Bank’s on -going efforts to 
provide better service to our customers and set higher standards for performance. The Policy 
is based on principles of transparency and fairness in the treatment of customers. 
 
The objective of the Policy is to establish a system where the Bank compensates the 
customer for any financial loss the customer might have incurred due to deficiency in service 
on the part of the Bank or any act of omission or commission directly attribut able to the 
Bank. 
 
This policy document covers the following aspects: 
1. Erroneous/Unauthorised debiting of accounts or fraudulent or other  transactions 
leading to financial loss to customers 
2. NACH direct debits/other debits instruction to accounts 
3. Credit Cards issued without customer’s consent 
4. Compensation payable on account of delays in collection

In [7]:
type(text)
len(text)
print(text[:500])
text = page.extract_text()

# current pipline is
"""
    PDF
        ↓
    PdfReader
        ↓
    PDF pages
        ↓
    page.extract_text()
        ↓
    Python string""" 

 
Page 1 of 18 
 
 
Approved by CSCM: March 20, 2026 
 
Customer Compensation Policy 
Introduction 
 
The Compensation Policy (the Policy) of the Bank reflects the Bank’s on -going efforts to 
provide better service to our customers and set higher standards for performance. The Policy 
is based on principles of transparency and fairness in the treatment of customers. 
 
The objective of the Policy is to establish a system where the Bank compensates the 
customer for any financial loss the custom


'\n    PDF\n        ↓\n    PdfReader\n        ↓\n    PDF pages\n        ↓\n    page.extract_text()\n        ↓\n    Python string'

LOADING embedding model 

In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [9]:
from pathlib import Path
from pypdf import PdfReader

PDF_FOLDER = Path("../requirement")

documents = []

for pdf_path in PDF_FOLDER.glob("*.pdf"):
    reader = PdfReader(pdf_path)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "text": text.strip(),
                "source": pdf_path.name,
                "page": page_number
            })

print("Pages extracted:", len(documents))

Pages extracted: 97


In [10]:
documents[0]

{'text': 'ICICI GROUP \nCODE OF BUSINESS CONDUCT AND \nETHICS \n \nApril 2025',
 'source': 'Code-of-Business.pdf',
 'page': 1}

CHUNKING!!

In [11]:
def chunk_text(text, chunk_size=1000, overlap=150):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

testing chunking    

In [12]:
test_chunks = chunk_text(documents[0]["text"])

print("Number of chunks:", len(test_chunks))
print(test_chunks[0])

Number of chunks: 1
ICICI GROUP 
CODE OF BUSINESS CONDUCT AND 
ETHICS 
 
April 2025


In [13]:
chunks = []

for document in documents:
    page_chunks = chunk_text(document["text"])

    for chunk_index, chunk in enumerate(page_chunks):
        chunks.append({
            "text": chunk,
            "source": document["source"],
            "page": document["page"],
            "chunk": chunk_index
        })

print("Total chunks:", len(chunks))

""" documents
    ↓
    pages
    ↓
    chunks"""

Total chunks: 279


' documents\n    ↓\n    pages\n    ↓\n    chunks'

CREATING EMBEDDING MODEL

In [14]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

(279, 384)


CREATING CHORMA DB

In [15]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="../data/chroma"
)


In [16]:
###CREATING A COLLECTION 

collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

### creating ids 

ids = [
    f"{chunk['source']}-{chunk['page']}-{chunk['chunk']}"
    for chunk in chunks
]

### creating meta data
#  
metadatas = [
    {
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk": chunk["chunk"]
    }
    for chunk in chunks
]

STORING EVERYTHING  

In [17]:
collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Stored:", collection.count())

"""  
PDF
    ↓
text
    ↓
chunks
    ↓
embeddings
    ↓
ChromaDB"""

Stored: 279


'  \nPDF\n    ↓\ntext\n    ↓\nchunks\n    ↓\nembeddings\n    ↓\nChromaDB'

In [18]:
question = "What is the compensation policy?"

### creating queries form question
query_embedding = embedding_model.encode(
    [question]
)[0]

### froming results form queris
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)


### getting results

results

{'ids': [['compensation-policy.pdf-1-0',
   'compensation-policy.pdf-2-0',
   'compensation-policy.pdf-17-1',
   'compensation-policy.pdf-18-0',
   'compensation-policy.pdf-18-1']],
 'embeddings': None,
 'documents': [['Page 1 of 18 \n \n \nApproved by CSCM: March 20, 2026 \n \nCustomer Compensation Policy \nIntroduction \n \nThe Compensation Policy (the Policy) of the Bank reflects the Bank’s on -going efforts to \nprovide better service to our customers and set higher standards for performance. The Policy \nis based on principles of transparency and fairness in the treatment of customers. \n \nThe objective of the Policy is to establish a system where the Bank compensates the \ncustomer for any financial loss the customer might have incurred due to deficiency in service \non the part of the Bank or any act of omission or commission directly attribut able to the \nBank. \n \nThis policy document covers the following aspects: \n1. Erroneous/Unauthorised debiting of accounts or fraudule

Top-K retrieval !!

In [19]:
for i in range(len(results["documents"][0])):
    print("=" * 80)

    print("SOURCE:", results["metadatas"][0][i]["source"])
    print("PAGE:", results["metadatas"][0][i]["page"])
    print("DISTANCE:", results["distances"][0][i])

    print(results["documents"][0][i])

### now we have
"""
Question
    ↓
Embedding
    ↓
Chroma
    ↓
5 closest chunks"""

SOURCE: compensation-policy.pdf
PAGE: 1
DISTANCE: 0.38204675912857056
Page 1 of 18 
 
 
Approved by CSCM: March 20, 2026 
 
Customer Compensation Policy 
Introduction 
 
The Compensation Policy (the Policy) of the Bank reflects the Bank’s on -going efforts to 
provide better service to our customers and set higher standards for performance. The Policy 
is based on principles of transparency and fairness in the treatment of customers. 
 
The objective of the Policy is to establish a system where the Bank compensates the 
customer for any financial loss the customer might have incurred due to deficiency in service 
on the part of the Bank or any act of omission or commission directly attribut able to the 
Bank. 
 
This policy document covers the following aspects: 
1. Erroneous/Unauthorised debiting of accounts or fraudulent or other  transactions 
leading to financial loss to customers 
2. NACH direct debits/other debits instruction to accounts 
3. Credit Cards issued without customer’s

'\nQuestion\n    ↓\nEmbedding\n    ↓\nChroma\n    ↓\n5 closest chunks'

adding threshold - to stop model form answering if it do not have knowledge

In [20]:
DISTANCE_THRESHOLD = 0.70

In [21]:
relevant_chunks = []

for i, distance in enumerate(results["distances"][0]):

    if distance <= DISTANCE_THRESHOLD:

        relevant_chunks.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "page": results["metadatas"][0][i]["page"],
            "distance": distance
        })

In [22]:
####checking 
# len(relevant_chunks)

In [24]:
for chunk in relevant_chunks:
    print(chunk["source"], chunk["page"], chunk["distance"])

###if no threshold

if not relevant_chunks:
    print("I couldn't find enough information in the knowledge base.")

compensation-policy.pdf 1 0.38204675912857056
compensation-policy.pdf 2 0.47630423307418823
compensation-policy.pdf 17 0.4844878911972046
compensation-policy.pdf 18 0.5009500980377197
compensation-policy.pdf 18 0.5209332704544067


### bilding LLM context 

In [25]:
context = "\n\n".join(
    f"""
Source: {chunk['source']}
Page: {chunk['page']}

{chunk['text']}
"""
    for chunk in relevant_chunks
)

print(context)

""" WHAT WE HAVE NOE
Chroma
    ↓
relevant chunks
    ↓
context
    ↓ 
LLM"""


Source: compensation-policy.pdf
Page: 1

Page 1 of 18 
 
 
Approved by CSCM: March 20, 2026 
 
Customer Compensation Policy 
Introduction 
 
The Compensation Policy (the Policy) of the Bank reflects the Bank’s on -going efforts to 
provide better service to our customers and set higher standards for performance. The Policy 
is based on principles of transparency and fairness in the treatment of customers. 
 
The objective of the Policy is to establish a system where the Bank compensates the 
customer for any financial loss the customer might have incurred due to deficiency in service 
on the part of the Bank or any act of omission or commission directly attribut able to the 
Bank. 
 
This policy document covers the following aspects: 
1. Erroneous/Unauthorised debiting of accounts or fraudulent or other  transactions 
leading to financial loss to customers 
2. NACH direct debits/other debits instruction to accounts 
3. Credit Cards issued without customer’s consent 
4. Compensation pa

' WHAT WE HAVE NOE\nChroma\n    ↓\nrelevant chunks\n    ↓\ncontext\n    ↓ \nLLM'

connecting whith llm model 

In [26]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL_NAME = os.getenv(
    "OPENROUTER_MODEL",
    "inclusionai/ling-3.0-flash-fin:free"
)

In [27]:
def ask_llm(question, context):

    prompt = f"""
You are a knowledge-base assistant.

Answer the user's question using ONLY the provided context.

If the context does not contain enough information to answer,
say:

"I couldn't find enough information in the knowledge base."

Context:
{context}

Question:
{question}
"""

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL_NAME,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        },
        timeout=60
    )

    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]

In [28]:
if relevant_chunks:

    answer = ask_llm(question, context)

    print(answer)

else:

    print("I couldn't find enough information in the knowledge base.")

Based on the provided context, the **Customer Compensation Policy** is a formal document approved by the Bank's Board of Management (on March 20, 2026) designed to improve service standards, based on principles of transparency and fairness. 

Its primary objective is to establish a system where the Bank compensates customers for financial losses incurred due to a deficiency in service, or any act of omission or commission directly attributable to the Bank.

The policy covers the following key aspects:
*   **Covered Issues:** Erroneous/unauthorized account debiting, fraudulent transactions, NACH direct debits, credit cards issued without customer consent, delays in collections, delayed credit of interest/maturity values (e.g., Floating Rate Savings Bonds), and delays in pension/arrears.
*   **Specific Compensation Rates:** 
    *   Delays in pension/arrears are compensated at a fixed rate of 8% per annum (credited automatically without the need for a claim).
    *   Delays in deceased c